# Which market feature subsets have spherical geometry?

**Hypothesis:** in a declared feature metric, an unnormalized subset (or unconstrained PCA coordinates of a subset) occupies a noisy spherical surface with meaningful angular coverage.

This experiment fits **surfaces**, not market regimes. No GMM, mixture distributions, clustering, or return forecasts are used. Radius is estimated from unnormalized coordinates; normalizing input rows would manufacture the property under investigation.

Workflow: chronological development folds → automatic subset/PCA search → repeated surrogate searches → affine/ellipsoidal controls → frozen final confirmation. A subset that fails is **not established as spherical**, not automatically Euclidean.

Edit the next cell to change the experiment. The mathematics is in `geometry.py`; selection and splitting are in `experiment.py`; practical iteration notes are in `EXPERIMENT_NOTES.md`.

In [1]:
from pathlib import Path
import sys, json, hashlib, time
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if not (ROOT / "data/market_features.csv").exists():
    ROOT = ROOT / "Thesis"
if not (ROOT / "data/market_features.csv").exists():
    raise FileNotFoundError("Set ROOT to the Thesis folder.")
sys.path.insert(0, str(ROOT))
from experiment import (make_candidates, chronological_splits, screen_candidates,
                        calibrate_search, attach_search_calibration,
                        compare_ellipsoid, final_evaluation)

FEATURES = ["DBC", "EEM", "EFA", "SPY", "SMB", "HML", "RMW", "CMA", "Mom",
            "d_10Y_Yield", "d_Yield_Curve", "d_Credit_Spread", "d_Inflation_Exp", "d_log_VIX"]
CONFIG = dict(
    features=FEATURES,
    seed_subsets={"ETF": FEATURES[:4], "Factors": FEATURES[4:9], "Macro": FEATURES[9:]},
    search_mode="seed_neighbors",  # "seeds", "seed_neighbors", or "exhaustive".
    subset_pool=FEATURES,
    subset_sizes=[3, 4],            # Only used by exhaustive mode.
    pca_dimensions=[None, 3],       # None = original subset; integers = training-fitted PCA.
    max_candidates=120,             # Refuse accidental combinatorial explosions.
    scaling="standard",            # "standard" or "none". Defines the geometry being tested.
    holdout_size=756,               # ~3 trading years, reserved before any search.
    n_splits=3,
    validation_size=252,
    gap=0,                         # No forward labels in this geometry objective.
    min_train_size=1000,
    fit_max_points=400,             # Seeded training subsample for surface optimization.
    sphere_max_nfev=150,            # Legacy key: now the SLSQP iteration budget.
    ellipsoid_max_nfev=100,
    max_radius_rms=2.0,             # Prevent a huge sphere from mimicking a plane.
    max_center_rms=1.5,
    max_resultant=0.60,             # Coverage check, not a distributional uniformity test.
    max_relative_shell_rmse=0.15,   # Held-out radial RMSE / fitted radius: declared noise tolerance.
    min_angular_span=0.05,          # q × smallest eigenvalue of mean(u uᵀ).
    ellipsoid_shortlist=3,          # More costly controls for the top eligible candidates.
    ellipsoid_margin=0.10,          # Allowed extra sphere MSE: 10% of plane MSE.
    null_families=["phase", "volatility_phase"],
    n_surrogates=19,                # Pilot resolution 1/(19+1)=0.05; use >=199 for stronger evidence.
    volatility_window=20,
    alpha=0.05,
    bootstrap_block=20,
    n_bootstrap=500,
    seed=42,
    progress_every=5,
)
RUN_FINAL_TEST = False             # Enable ONLY after freezing the development protocol.
EXPORT_RESULTS = True
BLAS_THREADS = 1
pd.set_option("display.max_colwidth", 85)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False})

## 1. Declare the search and reserve the final period

`seed_neighbors` automatically evaluates the seed blocks, each one-feature addition from `subset_pool`, and each one-feature deletion. It is a bounded search, not an exhaustive claim about every subset. `exhaustive` searches the configured subset sizes within a smaller pool. PCA is optional and fitted only on each training window; full-dimensional PCA is omitted because it only rotates the raw coordinates.

The input is a copied CSV in this folder. See `data/provenance.json` for the source checksum; historical release vintages have not been audited. Dates must be increasing and unique; missing values cause an explicit error.

In [2]:
source = ROOT / "data/market_features.csv"
data = pd.read_csv(source, index_col=0, parse_dates=True)[CONFIG["features"]].astype(float)
if not isinstance(data.index, pd.DatetimeIndex) or data.index.hasnans:
    raise ValueError("Invalid dates.")
if not data.index.is_unique or not data.index.is_monotonic_increasing:
    raise ValueError("Expected increasing, unique dates.")
if not np.isfinite(data.to_numpy()).all():
    raise ValueError("Resolve missing/nonfinite observations explicitly before analysis.")
h = CONFIG["holdout_size"]
if not 0 < h < len(data):
    raise ValueError("Invalid final-test size.")
development = data.iloc[:-h].copy()
HOLDOUT_START = data.index[-h]
candidates = make_candidates(CONFIG)
candidate_map = {c.name: c for c in candidates}
splits, split_table = chronological_splits(development, CONFIG)
assert all(development.index[va[-1]] < HOLDOUT_START for tr, va in splits)
display(split_table)
print(f"Automatic search: {len(candidates)} subset/representation candidates")
print(f"Reserved final period: {HOLDOUT_START.date()} through {data.index[-1].date()} ({h} observations)")
print("PCA candidates test their retained coordinates; they do not establish a spherical raw subset.")

Automatic search: 86 subset/representation candidates
Reserved final period: 2023-08-25 through 2026-08-31 (756 observations)
PCA candidates test their retained coordinates; they do not establish a spherical raw subset.


,fold,train_n,train_start,train_end,validation_start,validation_end
0,1,3498,2006-09-29,2020-08-21,2020-08-24,2021-08-23
1,2,3750,2006-09-29,2021-08-23,2021-08-24,2022-08-23
2,3,4002,2006-09-29,2022-08-23,2022-08-24,2023-08-24


## 2. Fit spheres and affine alternatives on training data

For a candidate with $q$ coordinates, compare a sphere $S^{q-1}$ to an affine hyperplane of dimension $q-1$. Both have codimension one. Held-out squared **orthogonal** distances are divided by training coordinate energy; all alternatives for that candidate use the same scale.

The search statistic is
$$T=\log\frac{\text{mean held-out plane loss}}{\text{mean held-out sphere loss}}.$$
Higher values indicate a better spherical approximation relative to a plane. This **alone is not evidence of spherical support**: ordinary high-dimensional clouds can also favor a shell. The next section calibrates the complete subset/PCA search using surrogate controls.

Sphere fitting estimates center and radius by geometric least squares. Bounded radius/center, thin-shell noise tolerance, and angular-coverage gates prevent an almost-flat giant sphere or a small cap from supporting a global spherical claim. These thresholds are explicit experimental assumptions, not universal laws. The plane and sphere use the same seeded training subsample; scaling/PCA use the full training window. No validation rows enter either fit.

In [3]:
started = time.perf_counter()
with threadpool_limits(limits=BLAS_THREADS):
    ranking, fit_diagnostics, validation_scores = screen_candidates(development, splits, candidates, CONFIG)
print(f"Screening completed in {time.perf_counter()-started:.1f}s")
display(ranking.head(12).round(4))
print("Fit-eligible candidates:", int(ranking.fit_ok.sum()))
print("Fit + coverage eligible:", int((ranking.fit_ok & ranking.coverage_ok).sum()))
issues = fit_diagnostics[(~fit_diagnostics.converged) | fit_diagnostics.boundary]
if len(issues):
    display(issues[["candidate", "fold", "converged", "boundary", "error"]].head(12))

Screening completed in 2.2s
Fit-eligible candidates: 0
Fit + coverage eligible: 0


,feature_count,coordinate_dimension,representation,statistic,sphere_loss,plane_loss,fit_ok,coverage_ok,thickness_ok,max_relative_shell_rmse,max_radial_cv,min_retained_variance
candidate,,,,,,,,,,,,
"CMA,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.8443,0.3208,0.7463,False,False,False,0.3880,0.3109,0.7111
"RMW,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.6652,0.3015,0.5864,False,False,False,0.3550,0.3013,0.7169
"SMB,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.5072,0.2609,0.4332,False,False,False,0.3460,0.3103,0.7071
"HML,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.3270,0.3218,0.4463,False,False,False,0.3573,0.3192,0.7041
"Mom,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.3171,0.2851,0.3914,False,False,False,0.3548,0.3190,0.7024
"SPY,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.2372,0.2226,0.2822,False,False,False,0.3481,0.3142,0.7929
"SMB,HML,RMW,Mom | PCA=3",4,3,PCA,0.1894,0.4765,0.5758,False,False,False,0.5016,0.3969,0.8775
"DBC,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",6,3,PCA,0.1877,0.2447,0.2952,False,False,False,0.3627,0.3201,0.7352
"EEM,SMB,HML,RMW,CMA,Mom | PCA=3",6,3,PCA,0.1815,0.4972,0.5962,False,False,False,0.4598,0.3784,0.7134


,candidate,fold,converged,boundary,error
0,"DBC,EEM,EFA | raw",1,True,True,
1,"DBC,EEM,EFA | raw",2,True,True,
2,"DBC,EEM,EFA | raw",3,True,True,
3,"DBC,EEM,SPY | raw",1,True,True,
4,"DBC,EEM,SPY | raw",2,True,True,
5,"DBC,EEM,SPY | raw",3,True,True,
6,"DBC,EFA,SPY | raw",1,True,True,
7,"DBC,EFA,SPY | raw",2,True,True,
8,"DBC,EFA,SPY | raw",3,True,True,
9,"EEM,EFA,SPY | raw",1,True,True,


## 3. Calibrate the whole search, not just the winner

Each surrogate repeats the **same** train-only preprocessing, subset/PCA search, splitting, and sphere/plane fitting. Record its maximum eligible search statistic. Comparing each observed candidate to these maxima accounts for searching the declared candidate set under that surrogate mechanism. Coverage filtering is omitted in the null maxima, which makes the gate conservative. A boundary fit is ineligible in both real and null searches; an unresolved numerical fit makes that surrogate maximum infinite rather than silently reducing the search.

Two predeclared controls are required:

- **Shared-phase surrogates:** preserve multivariate cross-spectra, means, and covariance while randomizing common Fourier phases.
- **Volatility-envelope surrogates:** remove a common trailing RMS envelope, randomize the residual phases, then restore the envelope. They approximately preserve common volatility timing; raw cross-spectra and marginal distributions are not preserved exactly.

The empirical values below are **conditional surrogate tail fractions**, not universal p-values for all Euclidean processes. Stationarity, marginal tails, cross-sectional nonlinear dependence, and the approximate envelope control limit interpretation. Rejecting both controls is necessary here, but still does not prove a sphere uniquely explains the data. Persistent volatility or nonstationarity can produce misleading surrogate results; examine period sensitivity.

Candidates with invalid constrained sphere fits have undefined tail fractions. With 19 repetitions, the minimum tail fraction is 0.05. This is a pilot configuration. Increase to 199 or more, freeze the null assumptions, and add diagnostics before thesis-level inference. Never normalize observations before these experiments.

In [4]:
started = time.perf_counter()
with threadpool_limits(limits=BLAS_THREADS):
    null_results = calibrate_search(development, splits, candidates, CONFIG)
ranking = attach_search_calibration(ranking, null_results)
print(f"Surrogate calibration completed in {time.perf_counter()-started:.1f}s")
display(null_results.groupby("family").agg(repetitions=("repeat", "count"),
                                            unresolved_searches=("unresolved", lambda s: int((s>0).sum())),
                                            median_eligible=("eligible_candidates", "median")))
p_columns = [c for c in ranking if c.startswith("p_")]
display(ranking[["statistic", "fit_ok", "coverage_ok", *p_columns]].head(12).round(4))

phase: 5/19 surrogate searches
phase: 10/19 surrogate searches
phase: 15/19 surrogate searches
phase: 19/19 surrogate searches
volatility_phase: 5/19 surrogate searches
volatility_phase: 10/19 surrogate searches
volatility_phase: 15/19 surrogate searches
volatility_phase: 19/19 surrogate searches
Surrogate calibration completed in 75.4s


,repetitions,unresolved_searches,median_eligible
family,,,
phase,19,0,34.0
volatility_phase,19,0,0.0


,statistic,fit_ok,coverage_ok,p_phase,p_volatility_phase
candidate,,,,,
"CMA,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.8443,False,False,NaN,NaN
"RMW,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.6652,False,False,NaN,NaN
"SMB,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.5072,False,False,NaN,NaN
"HML,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.3270,False,False,NaN,NaN
"Mom,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.3171,False,False,NaN,NaN
"SPY,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.2372,False,False,NaN,NaN
"SMB,HML,RMW,Mom | PCA=3",0.1894,False,False,NaN,NaN
"DBC,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.1877,False,False,NaN,NaN
"EEM,SMB,HML,RMW,CMA,Mom | PCA=3",0.1815,False,False,NaN,NaN


## 4. Ask whether the shape specifically needs equal radii

A sphere beating a plane might simply reveal curvature. Fit an ellipsoid control for the top `ellipsoid_shortlist` fit-and-coverage-eligible candidates. Its orientation is fixed to the training PCA axes; its center and positive semiaxes are optimized. It contains the sphere as a special case, but is not an unrestricted search over all orientations. Both fitting and scoring use true Euclidean closest-point distances, including observations inside the surface.

The sphere must beat the plane and be **noninferior** to the ellipsoid: the one-sided bootstrap upper bound on sphere-minus-ellipsoid error must be within `ellipsoid_margin × plane error`. The declared margin favors the simpler spherical shape when predictive errors are practically similar; it is not proof that semiaxes are equal. Bootstrap resampling stays within validation folds and is approximate under serial dependence/nonstationarity. Development intervals are descriptive after selection.

A supported development candidate must pass fit and coverage checks, the held-out shell-thickness tolerance, surrogate-search controls, the plane comparison, and the ellipsoid noninferiority gate. Among supported candidates, choose the highest search statistic. If none pass, the correct output is **no supported subset**, without forcing the remainder into a Euclidean category.

In [5]:
with threadpool_limits(limits=BLAS_THREADS):
    ranking, selected_name, comparison_scores, ellipsoid_diagnostics = compare_ellipsoid(
        development, splits, candidates, ranking, CONFIG)
print("Development selection:", selected_name or "No subset supported under this protocol")
columns = ["statistic", "fit_ok", "coverage_ok", "thickness_ok", *p_columns,
           "sphere_minus_plane_upper", "ellipsoid_noninferior", "ellipsoid_fit_ok", "candidate_supported"]
display(ranking[columns].head(15).round(4))

# Compact diagnostic plot; values are geometric approximation scores, not probabilities.
plot_rows = ranking[ranking.fit_ok].head(12).sort_values("statistic")
if len(plot_rows):
    ax = plot_rows.statistic.plot.barh(figsize=(10, 5))
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Validation log(plane MSE / sphere MSE): descriptive, before surrogate calibration")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No admissible spherical fits to plot. Inspect the radius/center boundary diagnostics above.")

Development selection: No subset supported under this protocol
No admissible spherical fits to plot. Inspect the radius/center boundary diagnostics above.


,statistic,fit_ok,coverage_ok,thickness_ok,p_phase,p_volatility_phase,sphere_minus_plane_upper,ellipsoid_noninferior,ellipsoid_fit_ok,candidate_supported
candidate,,,,,,,,,,
"CMA,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.8443,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"RMW,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.6652,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"SMB,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.5072,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"HML,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.3270,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"Mom,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.3171,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"SPY,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.2372,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"SMB,HML,RMW,Mom | PCA=3",0.1894,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"DBC,d_10Y_Yield,d_Yield_Curve,d_Credit_Spread,d_Inflation_Exp,d_log_VIX | PCA=3",0.1877,False,False,False,NaN,NaN,NaN,NaN,NaN,False
"EEM,SMB,HML,RMW,CMA,Mom | PCA=3",0.1815,False,False,False,NaN,NaN,NaN,NaN,NaN,False


## 5. Freeze the development record

Every run creates a timestamped directory containing the configuration, exact split dates, input/code hashes, candidate list, per-date validation losses, numerical diagnostics, surrogate-search maxima, ranking, and selected subset. Compare runs using a written hypothesis and one deliberate change at a time. Data and every newly generated artifact remain inside `Thesis`.

In [6]:
RUN_DIR = None
import platform, scipy, sklearn
protocol = dict(config=CONFIG, selected=selected_name, holdout_start=str(HOLDOUT_START),
                source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
                code_sha256={f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                             for f in ["geometry.py", "experiment.py"]},
                versions=dict(python=platform.python_version(), numpy=np.__version__,
                              pandas=pd.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__))
if EXPORT_RESULTS:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIR = ROOT / "results" / run_id
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR/"protocol.json").write_text(json.dumps(protocol, indent=2))
    pd.DataFrame([dict(candidate=c.name, columns=list(c.columns), dimension=c.dimension)
                  for c in candidates]).to_json(RUN_DIR/"candidates.json", orient="records", indent=2)
    split_table.to_csv(RUN_DIR/"splits.csv", index=False)
    ranking.to_csv(RUN_DIR/"ranking.csv")
    validation_scores.to_csv(RUN_DIR/"validation_scores.csv", index_label="date")
    fit_diagnostics.to_csv(RUN_DIR/"surface_fit_diagnostics.csv", index=False)
    comparison_scores.to_csv(RUN_DIR/"ellipsoid_comparison_scores.csv", index_label="date")
    ellipsoid_diagnostics.to_csv(RUN_DIR/"ellipsoid_fit_diagnostics.csv", index=False)
    null_results.to_csv(RUN_DIR/"surrogate_searches.csv", index=False)
    print("Saved:", RUN_DIR)
else:
    print("Export disabled.")

Saved: /Users/shashwatroy/VSCode Projects/Thesis/results/20261024T201757_924290Z


## 6. Untouched final confirmation — default off

Freeze the entire protocol before enabling `RUN_FINAL_TEST`. If development supports no subset, this cell does not consume the holdout. Otherwise refit that **single selected** coordinate transformation and set of surfaces on development data and evaluate fixed surfaces on the final period. The train-only metric and PCA are retained.

Final surrogate calibration evaluates only the frozen candidate against randomized versions of the final period; it does not rerun feature selection or refit on those surrogates. This is a conditional shape check, not a forecast simulation. The final result requires the predeclared coverage, plane, ellipsoid, and both surrogate gates. If you revise the protocol after seeing this result, this period is no longer an untouched confirmation sample.

In [7]:
final_summary = None
if not RUN_FINAL_TEST:
    print("Final period remains unevaluated. Freeze the protocol before enabling RUN_FINAL_TEST.")
elif selected_name is None:
    print("No development-supported subset. Final period remains unevaluated.")
else:
    with threadpool_limits(limits=BLAS_THREADS):
        final_scores, final_summary = final_evaluation(
            development, data.loc[HOLDOUT_START:], candidate_map[selected_name], CONFIG)
    display(pd.Series(final_summary, name="final_confirmation").to_frame())
    print("Supported under the declared controls:", final_summary["passes_confirmation"])
    if RUN_DIR is not None:
        final_scores.to_csv(RUN_DIR/"final_distances.csv", index_label="date")
        (RUN_DIR/"final_confirmation.json").write_text(json.dumps(final_summary, indent=2))

Final period remains unevaluated. Freeze the protocol before enabling RUN_FINAL_TEST.


## What to change in each iteration

| Iteration | Change | Keep fixed | Purpose |
|---|---|---|---|
| 0 | Run defaults | Input observations and final-test boundary | Establish the working search and diagnostics. |
| 1 | Raise `sphere_max_nfev`, `ellipsoid_max_nfev`, and `fit_max_points` | Candidate pool, metric, dates | Check numerical and training-subsample stability. |
| 2 | Change `search_mode`, `subset_pool`, or `subset_sizes` | Metric, PCA choices, chronology | Expand/restrict which raw subsets are tested. Rerun every surrogate search. |
| 3 | Change `pca_dimensions` | Candidate subsets and dates | Test unconstrained reduced coordinates, not imposed spherical latents. |
| 4 | Change the metric deliberately | Same subset search and dates | Check whether a conclusion depends on units/scaling. Report claims in that metric. |
| 5 | Increase `n_surrogates` and vary dependence settings | Hypothesis and selected search space | Improve calibration resolution and assess sensitivity to the control process. |
| 6 | Change development validation windows | Final boundary stays fixed | Check whether one market episode drives the result. |
| Final | Enable final confirmation | All scientific and numerical settings | Confirm the frozen candidate on an untouched period. |

Record hypothesis → one change → expected effect → diagnostics → validation result → next action. Do not optimize repeatedly against the final sample. A revised search is a new search and requires new calibration. Numerical failure is not evidence for an alternative geometry.

**Interpretation limits:** this notebook tests noisy shell geometry, not merely whether direction is useful with varying magnitude. It compares specific geometries and conditional surrogate controls; it cannot prove unique geometry or label every leftover feature linear. DR-supported coordinates establish a statement about the embedding, not the original subset. Local spherical patches, unions of spheres, unrestricted ellipsoids, and general manifolds require additional experiments.

Primary references: [spherical versus affine manifold approximation](https://arxiv.org/abs/1706.08263), [closest-point ellipsoid distances](https://geometrictools.com/Documentation/DistancePointEllipseEllipsoid.pdf), [multivariate surrogate time series](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.73.951), [surrogate-data assumptions and limitations](https://arxiv.org/abs/chao-dyn/9909037).